# LeanWatch on a free Colab GPU

This notebook shows how far LeanWatch's brain has come, using Google's free GPU instead of your laptop's CPU:

1. **Check the GPU** Colab gave you.
2. **Record practice demonstrations** on LeanWatch's practice computer (a web page driven by a headless browser).
3. **Measure the base model**: does it click inside the right element, and how fast does it answer on this GPU?
   The default is UI-TARS-2B-SFT, the model your 8 GB laptop runs with `--small`. On a T4 it first checks that the
   model works in float16 (Qwen2-VL models can overflow there) and switches to float32 if not.
4. **Fine-tune it** with LoRA. Checkpoints go to your Google Drive, so a lost session carries on where it stopped.
5. **Measure again**, see its clicks drawn on the screenshots, and keep the trained adapter.
6. **Print a report** to paste into Claude Code.

**Before you start:** *Runtime → Change runtime type → T4 GPU* (free). With Colab Pro, L4 or A100 GPUs are faster
and use bf16; the notebook detects them. Then *Runtime → Run all*. Colab asks once for access to your Google Drive.

With the default settings this takes roughly an hour on a T4; the report shows the real times. If a cell fails, run
the last cell (**Report**) anyway and paste its output to Claude. When you are done, *Runtime → Disconnect and
delete runtime* so you don't use up your free GPU time.

*Why a GPU and not Colab's free TPU? This pipeline (PyTorch, PEFT and TRL with a vision model) is built and tested
for NVIDIA GPUs.*

In [ ]:
#@title 1 · Settings { display-mode: "form" }
#@markdown The defaults are a quick progress test on a free T4. Your 8 GB laptop can only run the 2B model.
BASE_MODEL = "ByteDance-Seed/UI-TARS-2B-SFT"  #@param ["ByteDance-Seed/UI-TARS-2B-SFT", "ByteDance-Seed/UI-TARS-1.5-7B"]
#@markdown Practice episodes to record (each has about 4 screenshots and takes a few seconds):
EPISODES = 300  #@param {type:"integer"}
#@markdown Examples to measure, before and after training:
EVAL_EXAMPLES = 100  #@param {type:"integer"}
#@markdown Training steps (8 examples each). Raise it later and run again: training carries on from the last checkpoint.
TRAIN_STEPS = 100  #@param {type:"integer"}
#@markdown Screenshot size cap in pixels: 1003520 (default) or 602112 (faster on your laptop's CPU, less detail). The adapter remembers it.
MAX_PIXELS = 1003520  #@param {type:"integer"}
#@markdown GPU precision. "auto" checks first whether the model works in float16 on this GPU (Qwen2-VL models can overflow) and falls back to float32:
GPU_DTYPE = "auto"  #@param ["auto", "float16", "bfloat16", "float32"]
#@markdown Keep checkpoints, the adapter and the report in Google Drive (MyDrive/LeanWatch):
SAVE_TO_DRIVE = True  #@param {type:"boolean"}
#@markdown Optional: also upload the adapter to this **private** Hugging Face repo, e.g. `your-name/uitars2b-leanwatch`. Needs a WRITE token saved as the Colab secret `HF_TOKEN` (key icon on the left).
HF_REPO = ""  #@param {type:"string"}
BRANCH = "claude/bold-bell-tq6s9b"  #@param {type:"string"}

import json
import os
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
TEST_MODE = os.environ.get("LEANWATCH_NOTEBOOK_TEST") == "1"   # Claude's offline test of this notebook: leave unset
REPO = Path(os.environ.get("LEANWATCH_REPO", "/content/lean-watch"))
WORK = Path(os.environ.get("LEANWATCH_WORK", "/content/work"))
WORK.mkdir(parents=True, exist_ok=True)
MAX_NEW_TOKENS, VAL_FRACTION = 256, 0.1
if TEST_MODE:   # a tiny random model and a few examples, on a CPU
    EPISODES, EVAL_EXAMPLES, TRAIN_STEPS, MAX_PIXELS, SAVE_TO_DRIVE, HF_REPO = 8, 4, 2, 28 * 28 * 48, False, ""
    MAX_NEW_TOKENS, VAL_FRACTION = 8, 0.4
    sys.path.insert(0, str(REPO / "training" / "tests"))
    from tiny_model import build as build_tiny
    BASE_MODEL = str(build_tiny(WORK / "tiny", family="qwen2_vl"))

DATA = WORK / "data"
KEEP = Path("/content/drive/MyDrive/LeanWatch") if SAVE_TO_DRIVE and IN_COLAB else WORK / "keep"
NAME = BASE_MODEL.rstrip("/").split("/")[-1].lower() + "-leanwatch"
RUN = KEEP / "runs" / NAME                  # the adapter, plus checkpoints while training
RESULTS = KEEP / "results"
LOGS = KEEP / "logs"
times = {}


def sh(cmd, log=None, cwd=None):
    """Runs a shell command, shows its output as it comes, appends it to `log`, and stops on failure."""
    print("$", cmd, flush=True)
    t0 = time.monotonic()
    out = open(log, "a") if log else None
    p = subprocess.Popen(cmd, shell=True, cwd=cwd or (REPO if REPO.exists() else None), stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, env={**os.environ, "PYTHONUNBUFFERED": "1"})
    while chunk := p.stdout.read1(65536):
        text = chunk.decode(errors="replace")
        sys.stdout.write(text)
        sys.stdout.flush()
        if out:
            out.write(text)
    code = p.wait()
    if out:
        out.close()
    if code:
        raise RuntimeError(f"That command failed (exit code {code}). Run the Report cell at the end and paste it to Claude.")
    return round(time.monotonic() - t0, 1)


print("Model:", BASE_MODEL, "| episodes:", EPISODES, "| eval examples:", EVAL_EXAMPLES, "| training steps:", TRAIN_STEPS)

In [ ]:
#@title 2 · Check the GPU
def query(fields):
    r = subprocess.run(f"nvidia-smi --query-gpu={fields} --format=csv,noheader,nounits", shell=True,
                       capture_output=True, text=True)
    return r.stdout.strip().splitlines()[0].split(", ") if r.returncode == 0 and r.stdout.strip() else None


GPU = None
found = query("name,memory.total")
if found:
    cap = query("compute_cap")
    GPU = {"name": found[0], "vram_gb": round(float(found[1]) / 1024, 1), "compute_cap": cap[0] if cap else "?"}
    print(f"GPU: {GPU['name']}, {GPU['vram_gb']} GB, compute capability {GPU['compute_cap']}")
    if GPU["compute_cap"] != "?":
        print("Training precision:", "bf16" if float(GPU["compute_cap"]) >= 8 else "fp16 (this GPU has no fast bf16)")
elif TEST_MODE:
    print("No GPU (offline test): everything runs on the CPU.")
else:
    raise RuntimeError("There is no GPU in this session. Runtime → Change runtime type → T4 GPU, then Runtime → Run all.")

FOUR_BIT = "7B" in BASE_MODEL and (GPU is None or GPU["vram_gb"] < 40)
if "7B" in BASE_MODEL:
    print("Note: a 7B adapter can't run on an 8 GB laptop. Use it to compare; train the 2B for LeanWatch.")
    if FOUR_BIT:
        print("The 7B loads in 4-bit (QLoRA) to fit this GPU.")
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
disk = shutil.disk_usage(WORK).free / 1024**3
print(f"RAM {ram:.1f} GB, free disk {disk:.0f} GB")

In [ ]:
#@title 3 · Get LeanWatch's code (and connect Google Drive)
if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
for folder in (DATA, RUN.parent, RESULTS, LOGS):
    folder.mkdir(parents=True, exist_ok=True)

if TEST_MODE:
    print("Offline test: using the code in", REPO)
elif (REPO / ".git").exists():
    sh(f"git fetch -q origin {BRANCH} && git checkout -q {BRANCH} && git pull -q --ff-only origin {BRANCH}")
else:
    sh(f"git clone -q -b {BRANCH} https://github.com/bluestonerealty2026-a11y/lean-watch {REPO}", cwd="/content")
COMMIT = subprocess.run("git rev-parse --short HEAD", shell=True, cwd=REPO, capture_output=True, text=True).stdout.strip()
sys.path.insert(0, str(REPO / "agent"))
print("LeanWatch code at commit", COMMIT)

In [ ]:
#@title 4 · Install the training libraries and the recorder's browser
#@markdown Keeps Colab's own CUDA build of PyTorch. Takes a few minutes.
def node_major():
    r = subprocess.run("node --version", shell=True, capture_output=True, text=True)
    m = re.match(r"v(\d+)", r.stdout.strip())
    return int(m.group(1)) if m else 0


if not TEST_MODE:
    packages = '"transformers>=5.2" "accelerate>=1.1.0" "peft>=0.14" "trl>=1.0" "datasets>=3.0" "pillow>=10.0.1" psutil'
    sh(f"pip install -q --upgrade {packages}" + (" bitsandbytes" if GPU else ""), log=LOGS / "install.log")
    # Colab ships an old torchao, and PEFT refuses to add LoRA layers while it is installed. Nothing here uses it.
    sh("pip uninstall -y -q torchao", log=LOGS / "install.log")
    if node_major() < 20:   # the recorder's Playwright needs Node.js 20 or newer
        import hashlib
        import urllib.request

        base = "https://nodejs.org/dist/latest-v22.x/"
        sums = urllib.request.urlopen(base + "SHASUMS256.txt").read().decode()
        digest, tarball = next(line.split() for line in sums.splitlines() if line.endswith("-linux-x64.tar.xz"))
        blob = urllib.request.urlopen(base + tarball).read()
        assert hashlib.sha256(blob).hexdigest() == digest, "Node.js download doesn't match its checksum"
        Path("/content/node.tar.xz").write_bytes(blob)
        sh("mkdir -p /content/node && tar -xJf /content/node.tar.xz -C /content/node --strip-components=1", cwd="/content")
        os.environ["PATH"] = "/content/node/bin:" + os.environ["PATH"]
    sh("npm ci --no-audit --no-fund && npx playwright install --with-deps chromium", log=LOGS / "install.log")
sh('python -c "import torch, transformers, peft, trl; print(\'torch\', torch.__version__, \'| CUDA\', torch.version.cuda, '
   '\'| transformers\', transformers.__version__, \'| peft\', peft.__version__, \'| trl\', trl.__version__)"')
print("Node.js", subprocess.run("node --version", shell=True, capture_output=True, text=True).stdout.strip())

In [ ]:
#@title 5 · Record practice demonstrations
#@markdown The same seed always records the same episodes, so after a lost session this gives the same data again.
practice = DATA / "practice"
meta_file = practice / "meta.json"
if meta_file.exists() and json.loads(meta_file.read_text()).get("episodes") == EPISODES:
    print("Already recorded:", practice)
else:
    shutil.rmtree(practice, ignore_errors=True)
    times["record_s"] = sh(f"node training/record_practice.mjs --episodes {EPISODES} --out {practice} --seed 1 --scale 1.5",
                           log=LOGS / "record.log")
recorded = json.loads(meta_file.read_text())
size_mb = sum(f.stat().st_size for f in practice.rglob("*") if f.is_file()) / 1e6
print(f"{recorded['episodes']} episodes, {recorded['steps']} steps, {recorded['grounding']} grounding samples, {size_mb:.0f} MB"
      + (f", {times['record_s'] / max(1, EPISODES):.1f} s per episode" if "record_s" in times else ""))

from IPython.display import display
from PIL import Image

first = sorted((practice / "images").glob("*.png"))[0]
shot = Image.open(first)
shot.thumbnail((900, 900))
display(shot)

In [ ]:
#@title 6 · Build the training set
from transformers import AutoConfig

from leanwatch.local_model import coords_for

COORDS = coords_for(AutoConfig.from_pretrained(BASE_MODEL).model_type)   # rel1000 for UI-TARS-2B, abs for UI-TARS-1.5
sft = DATA / "sft"
shutil.rmtree(sft, ignore_errors=True)
sh(f"python training/build_dataset.py --practice {practice} --out {sft} --coords {COORDS} --max-pixels {MAX_PIXELS} "
   f"--val-fraction {VAL_FRACTION}", log=LOGS / "build.log")
built = json.loads((sft / "meta.json").read_text())
print("Coordinates:", COORDS, "| counts:", built["counts"])
if not sum(built["counts"]["val"].values()):
    raise RuntimeError("No examples were set aside for measuring: raise EPISODES in cell 1 and run again.")

In [ ]:
#@title 7 · Check the precision on this GPU
#@markdown The first run downloads the model (about 4.5 GB for the 2B). On a T4 (no fast bf16) it tries float16 on a
#@markdown few examples: if the answers come out unreadable (e.g. "!!!!"), it uses float32 instead.
FOUR = " --4bit" if FOUR_BIT else ""
PRECISION_NOTE = "set by hand"
if GPU_DTYPE == "auto" and GPU and GPU["compute_cap"] != "?" and float(GPU["compute_cap"]) < 8:
    probe = RESULTS / f"{NAME}-probe.jsonl"
    probe.unlink(missing_ok=True)
    try:
        sh(f"python training/evaluate.py --data {sft} --split val --limit 6 --model {BASE_MODEL} --max-new-tokens 64 "
           f"--gpu-dtype float16 --save {probe}{FOUR}", log=LOGS / "eval.log")
        answers = [json.loads(line) for line in open(probe)]
    except RuntimeError:
        answers = []
    readable = sum(r["parsed"] for r in answers)
    GPU_DTYPE = "float16" if readable else "float32"
    PRECISION_NOTE = (f"float16 probe: {readable}/{len(answers)} readable answers, e.g. {answers[0]['answer'][:60]!r}"
                      if answers else "float16 probe crashed")
elif GPU_DTYPE == "auto":
    PRECISION_NOTE = "native bf16 on this GPU" if GPU else "CPU"
# A 2B model in float32 doesn't leave room to train on a 16 GB GPU, so training loads it in 4-bit (QLoRA, float32 maths).
TRAIN_FOUR = FOUR_BIT or bool(GPU and GPU_DTYPE == "float32" and GPU["vram_gb"] < 24)
DTYPE = f" --gpu-dtype {GPU_DTYPE}"
print(f"Precision: {GPU_DTYPE} ({PRECISION_NOTE}); training in 4-bit: {TRAIN_FOUR}")

In [ ]:
#@title 8 · Measure the base model (before training)
def summarize(path):
    rows = [json.loads(line) for line in open(path)] if Path(path).exists() else []
    out = {}
    for task in ("grounding", "actions"):
        rs = [r for r in rows if r["task"] == task]
        if rs:
            points = [r["point_ok"] for r in rs if r["point_ok"] is not None]
            out[task] = {"examples": len(rs), "readable answer": sum(r["parsed"] for r in rs) / len(rs),
                         "right action": sum(r["type_ok"] for r in rs) / len(rs),
                         "click inside target": sum(points) / len(points) if points else None}
    if rows:
        out["seconds per answer"] = sum(r.get("seconds", 0) for r in rows) / len(rows)
    return out


def comparison(a, b):
    lines = [f"{'':34s}{'before':>10s}{'after':>10s}"]
    for task in ("grounding", "actions"):
        for metric in ("readable answer", "right action", "click inside target"):
            x, y = a.get(task, {}).get(metric), b.get(task, {}).get(metric)
            if x is not None or y is not None:
                cell = lambda v: f"{v:10.1%}" if v is not None else f"{'-':>10s}"
                lines.append(f"{task + ': ' + metric:34s}{cell(x)}{cell(y)}")
    sa, sb = a.get("seconds per answer"), b.get("seconds per answer")
    lines.append(f"{'seconds per answer on this GPU':34s}" + "".join(f"{v:10.1f}" if v is not None else f"{'-':>10s}" for v in (sa, sb)))
    return "\n".join(lines)


before = RESULTS / f"{NAME}-before.jsonl"
after = RESULTS / f"{NAME}-after.jsonl"
before.unlink(missing_ok=True)
times["eval_before_s"] = sh(f"python training/evaluate.py --data {sft} --split val --limit {EVAL_EXAMPLES} --model {BASE_MODEL} "
                            f"--max-new-tokens {MAX_NEW_TOKENS} --save {before}{DTYPE}{FOUR}", log=LOGS / "eval.log")
print(comparison(summarize(before), {}))

In [ ]:
#@title 9 · Fine-tune with LoRA
#@markdown Saves a checkpoint every few steps (to Drive). If the session ends, run everything again: it carries on.
SAVE_EVERY = max(1, min(25, TRAIN_STEPS // 4))
train_log = LOGS / "train.log"
start = train_log.stat().st_size if train_log.exists() else 0
times["train_s"] = sh(f"python training/train_sft.py --data {sft} --base {BASE_MODEL} --out {RUN} --max-steps {TRAIN_STEPS} "
                      f"--save-steps {SAVE_EVERY} --resume{DTYPE}" + (" --4bit" if TRAIN_FOUR else ""), log=train_log)
with open(train_log, errors="replace") as f:
    f.seek(start)
    losses = re.findall(r"""['"]loss['"]:\s*['"]?([\d.]+|nan|inf)""", f.read())
if any(v in ("nan", "inf") for v in losses):
    print(f"\nWARNING: the loss became nan/inf, so this adapter is broken (the numbers overflowed in {GPU_DTYPE}). "
          f"Delete the folder {RUN}, set GPU_DTYPE to float32 in cell 1, and run everything again.")
elif losses:
    print(f"\nLoss went from {losses[0]} to {losses[-1]} ({len(losses)} values logged).")

In [ ]:
#@title 10 · Measure again (after training)
after.unlink(missing_ok=True)
times["eval_after_s"] = sh(f"python training/evaluate.py --data {sft} --split val --limit {EVAL_EXAMPLES} --adapter {RUN} "
                           f"--max-new-tokens {MAX_NEW_TOKENS} --save {after}{DTYPE}{FOUR}", log=LOGS / "eval.log")
print(comparison(summarize(before), summarize(after)))

In [ ]:
#@title 11 · See the clicks (green: the right element · orange: before training · teal: after)
from IPython.display import display
from PIL import Image, ImageDraw

from leanwatch.actions import parse_response


def clicked(rec):
    """Where the answer clicks, in the screenshot's own pixels (None if it doesn't click)."""
    try:
        point = parse_response(rec["answer"]).actions[0].point
    except Exception:
        return None
    if point is None:
        return None
    w, h = rec["meta"]["width"], rec["meta"]["height"]
    if COORDS == "rel1000":
        return point[0] * w / 1000, point[1] * h / 1000
    iw, ih = rec["image_size"]
    return point[0] * w / iw, point[1] * h / ih


def mark(draw, xy, colour, f):
    if xy:
        x, y = xy[0] * f, xy[1] * f
        draw.ellipse((x - 14, y - 14, x + 14, y + 14), outline=colour, width=5)
        draw.line((x - 22, y, x + 22, y), fill=colour, width=3)
        draw.line((x, y - 22, x, y + 22), fill=colour, width=3)


rows_before = [json.loads(line) for line in open(before)] if before.exists() else []
rows_after = [json.loads(line) for line in open(after)] if after.exists() else []
shown = 0
for rec_b, rec_a in zip(rows_before, rows_after):
    if "bbox" not in rec_a["meta"] or shown >= 6:
        continue
    img = Image.open(rec_a["image"]).convert("RGB")
    f = img.width / rec_a["meta"]["width"]
    draw = ImageDraw.Draw(img)
    draw.rectangle([v * f for v in rec_a["meta"]["bbox"]], outline="#22c55e", width=5)
    mark(draw, clicked(rec_b), "#f59e0b", f)
    mark(draw, clicked(rec_a), "#3BBFB3", f)
    verdict = lambda r: "inside" if r["point_ok"] else ("outside" if r["point_ok"] is False else "no click")
    print(f"{rec_a['task']}: {rec_a['meta'].get('label', '')!r}   before: {verdict(rec_b)}   after: {verdict(rec_a)}")
    img.thumbnail((900, 900))
    display(img)
    shown += 1
if not shown:
    print("Nothing to show yet: cells 8 and 10 write the answers.")

In [ ]:
#@title 12 · Keep the trained adapter
#@markdown In Drive it is already kept. Without Drive it downloads as a zip. With HF_REPO set it also goes to that private repo.
kept = WORK / "adapter" / NAME
shutil.rmtree(kept, ignore_errors=True)
shutil.copytree(RUN, kept, ignore=shutil.ignore_patterns("checkpoint-*"))
if SAVE_TO_DRIVE and IN_COLAB:
    print(f"Kept in Google Drive: MyDrive/LeanWatch/runs/{NAME}")
elif IN_COLAB:
    from google.colab import files

    files.download(shutil.make_archive(str(WORK / NAME), "zip", kept))
if HF_REPO and not TEST_MODE:
    from google.colab import userdata
    from huggingface_hub import HfApi

    try:
        token = userdata.get("HF_TOKEN")
    except Exception as err:
        raise RuntimeError("Add a WRITE token as the Colab secret HF_TOKEN (key icon on the left) and allow this "
                           "notebook to use it.") from err
    api = HfApi(token=token)
    api.create_repo(HF_REPO, private=True, exist_ok=True)
    api.upload_folder(folder_path=str(kept), repo_id=HF_REPO, commit_message="LeanWatch adapter from Colab")
    print(f"Uploaded to the private repo https://huggingface.co/{HF_REPO}")

if HF_REPO:
    source = f"hf download {HF_REPO} --local-dir ~/lw-runs/{NAME}"
elif SAVE_TO_DRIVE:
    source = f"download MyDrive/LeanWatch/runs/{NAME} from drive.google.com, without its checkpoint-* folders, to ~/lw-runs/{NAME}"
else:
    source = f"unzip the downloaded {NAME}.zip into ~/lw-runs/{NAME}"
print(f"""
On your laptop:
  {source}
  cd ~/lean-watch/agent && conda activate leanwatch
  python -m leanwatch --adapter ~/lw-runs/{NAME} --dry-run      # no --4bit on the laptop
""")

In [ ]:
#@title 13 · Report for Claude (copy everything this cell prints into Claude Code)
def tail(path, n):
    return Path(path).read_text(errors="replace").splitlines()[-n:] if Path(path).exists() else []


report = ["LeanWatch Colab report",
          f"commit {globals().get('COMMIT', '?')}  model {BASE_MODEL}  episodes {EPISODES}  eval {EVAL_EXAMPLES}  "
          f"steps {TRAIN_STEPS}  max_pixels {MAX_PIXELS}  4bit {globals().get('FOUR_BIT')}",
          f"GPU {globals().get('GPU')}"]
versions = subprocess.run('python -c "import torch, transformers, peft, trl; print(torch.__version__, torch.version.cuda, '
                          'transformers.__version__, peft.__version__, trl.__version__)"',
                          shell=True, capture_output=True, text=True)
report.append("torch / CUDA / transformers / peft / trl: " + (versions.stdout.strip() or versions.stderr.strip()[-300:]))
report.append(f"times (s): {times}")
for label, path in (("recorded", DATA / "practice" / "meta.json"), ("dataset", DATA / "sft" / "meta.json")):
    if path.exists():
        m = json.loads(path.read_text())
        report.append(f"{label}: " + json.dumps({k: m[k] for k in ("episodes", "steps", "grounding", "counts", "coords") if k in m}))
report.append(f"precision: {globals().get('GPU_DTYPE')} ({globals().get('PRECISION_NOTE', 'not checked')}); "
              f"training in 4-bit: {globals().get('TRAIN_FOUR')}")
if "comparison" in globals():
    report.append(comparison(summarize(RESULTS / f"{NAME}-before.jsonl"), summarize(RESULTS / f"{NAME}-after.jsonl")))
train_log = LOGS / "train.log"
if train_log.exists():
    text = train_log.read_text(errors="replace")
    losses = re.findall(r"""['"]loss['"]:\s*['"]?([\d.]+|nan|inf)""", text)
    if losses:
        report.append(f"training loss: first {losses[0]}, last {losses[-1]} ({len(losses)} logged)")
    report += [line for line in text.splitlines() if "train_runtime" in line or "saved adapter" in line
               or "carrying on" in line or "training on" in line][-6:]
for log in sorted(LOGS.glob("*.log")):
    if "Traceback" in log.read_text(errors="replace"):
        report += [f"--- end of {log.name} ---"] + tail(log, 30)
text = "\n".join(report)
(KEEP / "report.txt").write_text(text)
print(text)